# Ingest a PDF and inspect the parser output

Before indexing full papers, check what the parser actually extracted. This optional notebook uses the two open sample PDFs in `pdf_samples/` and the included `pdfminer` loader. It needs no model download or external service.

In [ ]:
import sys
from pathlib import Path
from textwrap import shorten

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb

pdf_paths = sorted(nb.pdf_samples_dir().glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError("No sample PDFs found in notebooks/pdf_samples/")
work_dir = nb.bootstrap()
print([path.name for path in pdf_paths])

## 1. Parse and store the papers

The loader returns sections, metadata, and references. `InMemoryAcademicDB` stores these under a named strategy and saves a JSON backup so the records can be reopened later.

In [ ]:
from epilens.db.in_memory_academic_db import InMemoryAcademicDB
from epilens.rag.ingestion.document_loader import DocumentLoaderFactory

strategy = "sample-pdfs"
loader = DocumentLoaderFactory.get_loader("pdfminer")
db_path = work_dir / "academic_db.json"
db = InMemoryAcademicDB(backup_file=str(db_path))

for path in pdf_paths:
    sections, metadata, references = loader.load(path)
    paper_id = path.stem
    metadata.file_path = str(path)
    metadata_record = metadata.to_dict()
    metadata_record["original_paper_id"] = paper_id
    db.insert(paper_id, "sections", strategy, [s.to_dict() for s in sections])
    db.insert(paper_id, "metadata", strategy, metadata_record)
    db.insert(paper_id, "references", strategy, [r.to_dict() for r in references])

print("Stored paper IDs:", db.list_docs(strategy))

## 2. Check the extraction before retrieval

A successful parse does not guarantee good paper structure. The lightweight loader can fall back to a filename for the title, split one article into many fragments, or return no references. These are visible quality checks, not ingestion errors.

In [ ]:
for paper_id in db.list_docs(strategy):
    metadata = db.retrieve(paper_id, "metadata", strategy)
    sections = db.retrieve(paper_id, "sections", strategy) or []
    references = db.retrieve(paper_id, "references", strategy) or []
    print(f"{paper_id}: title={metadata.title!r}, sections={len(sections)}, references={len(references)}")

Look at several substantial sections, not just the first page. If the section boundaries are poor, later retrieval and classification can inherit that problem.

In [ ]:
paper_id = db.list_docs(strategy)[0]
sections = db.retrieve(paper_id, "sections", strategy) or []
for section in sorted(sections, key=lambda item: len(item.content), reverse=True)[:3]:
    preview = shorten(" ".join(section.content.split()), width=250)
    print(f"{section.title or '(untitled)'} [{section.section_type}]")
    print(" ", preview)

## 3. Reopen the local store

The same strategy name retrieves the records after reopening the backup file.

In [ ]:
reopened = InMemoryAcademicDB(backup_file=str(db_path))
print(reopened.list_docs(strategy))

**Try it:** compare these section previews with the original PDFs. If you need better layout or reference extraction for your own corpus, use an appropriate parser such as `unstructured` or a running GROBID service, then recheck a sample before indexing.